# small-llm — train a tiny GPT in Colab

Clone the repo, install deps, train (locally or by **streaming web-scale data**), generate with a KV cache, run **4-bit quantized** inference, and **build on an open-weight model** (fine-tune + grow).

**Tip:** *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*.

## 1. Clone & install

In [ ]:
REPO_URL = "https://github.com/lukas04545/llm.git"  # change to your fork if needed
import os
if not os.path.isdir("llm"):
    !git clone $REPO_URL llm
%cd llm
!pip install -q torch requests
# Optional extras (uncomment what you need):
# !pip install -q datasets tiktoken   # web streaming + GPT-2 BPE
# !pip install -q bitsandbytes        # FP4 / NF4 4-bit quantization (GPU)
# !pip install -q torchao             # int4 quantization (CPU + GPU)
# !pip install -q transformers        # import open-weight models (section 5)

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2A. Option A — train on a small local corpus
Downloads Tiny Shakespeare (offline fallback included).

In [ ]:
!python scripts/prepare_data.py
import torch
if torch.cuda.is_available():
    !python train.py --device=cuda --dtype=bf16 --n_layer=6 --n_head=6 --n_embd=384 \
        --block_size=256 --batch_size=64 --grad_accum_steps=2 --max_iters=5000
else:
    !python train.py

## 2B. Option B — train on web-scale streaming data (no download)
Streams FineWeb and tokenizes on the fly with GPT-2 BPE. Needs `datasets` + `tiktoken` (install them in cell 1). Memory stays flat regardless of corpus size.

In [ ]:
# !pip install -q datasets tiktoken
# !python train.py --stream=fineweb --tokenizer=gpt2 --device=cuda --dtype=bf16 \
#     --n_layer=6 --n_head=6 --n_embd=384 --block_size=256 \
#     --batch_size=32 --grad_accum_steps=4 --grad_checkpoint=true --max_iters=20000

## 3. Generate (KV cache on by default)

In [ ]:
!python generate.py --prompt="To be, or not to be" --max_new_tokens=400 --temperature=0.8 --top_k=40

## 4. Quantize to 4-bit and benchmark
On GPU, `nf4`/`fp4` need bitsandbytes; on CPU use `int8` (no extra deps).

In [ ]:
import torch
if torch.cuda.is_available():
    !pip install -q bitsandbytes
    !python bench.py --quantize=none --device=cuda --max_new_tokens=256
    !python bench.py --quantize=nf4  --device=cuda --max_new_tokens=256
else:
    !python bench.py --quantize=none --max_new_tokens=128
    !python bench.py --quantize=int8 --max_new_tokens=128

## 5. Build on an open-weight model — import, fine-tune, grow
Import a pretrained Llama-family model (SmolLM-135M), fine-tune it on the Shakespeare data, grow it deeper (identity-preserving), and train again. Uses a separate `out_hf/` dir so it won't overwrite the model above. A GPU is recommended.

In [ ]:
!pip install -q transformers
# 1) Import an open-weight model -> out_hf/
!python scripts/import_hf.py --repo=HuggingFaceTB/SmolLM-135M --out_dir=out_hf
!python generate.py --out_dir=out_hf --prompt="The meaning of life is" --max_new_tokens=80

In [ ]:
import torch
dev = "cuda" if torch.cuda.is_available() else "cpu"
!python scripts/prepare_data.py
# 2) Fine-tune the pretrained model on your data (small LR, reuses its tokenizer)
!python train.py --init_from=out_hf/ckpt.pt --out_dir=out_hf --data_path=data/input.txt \
    --device=$dev --learning_rate=2e-5 --batch_size=4 --block_size=128 --max_iters=100
# 3) Grow it deeper (identity-preserving), then continue training
!python scripts/grow.py --add=4 --ckpt=out_hf/ckpt.pt
!python train.py --init_from=out_hf/ckpt.pt --out_dir=out_hf --data_path=data/input.txt \
    --device=$dev --learning_rate=2e-5 --batch_size=4 --block_size=128 --max_iters=100
!python generate.py --out_dir=out_hf --prompt="ROMEO:" --max_new_tokens=120

## 6. (Optional) Generate from Python

In [ ]:
import torch
from generate import load_model
device = "cuda" if torch.cuda.is_available() else "cpu"
model, tok = load_model("out", device, quantize="none")
prompt = "What's in a name?"
idx = torch.tensor([tok.encode(prompt)], dtype=torch.long, device=device)
out = model.generate(idx, max_new_tokens=300, temperature=0.8, top_k=40)
print(tok.decode(out[0].tolist()))